In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import  LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score



# Load the datasets
housing = pd.read_csv('housing.csv')

housing['income_cat'] = pd.cut(housing['median_income'], bins=[0.0,1.5,3.0,4.5,6.0, np.inf], labels=[1,2,3,4,5])

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=0)

for train_index, test_index in split.split(housing, housing['income_cat']):
    start_train_set = housing.iloc[train_index].drop('income_cat', axis=1)
    start_test_set = housing.iloc[test_index].drop('income_cat', axis=1)

# we will work on the training data

housing = start_train_set.copy()

# 3. Separate predictors and labels
housing_labels = housing["median_house_value"].copy()
housing = housing.drop("median_house_value", axis=1)

# 4. Separate numerical and categorical columns
num_attribs = housing.drop("ocean_proximity", axis=1).columns.tolist()
cat_attribs = ["ocean_proximity"]


# 5 Lets make the pipeline for numerical and categoricals

# for Numericals
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("std_scaler", StandardScaler()),
])

# for categorical
cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# Construct the full pipeline
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])
# 6 Transform the data

housing_prepared = full_pipeline.fit_transform(housing)

print(housing_prepared)

# 7 Training Model

# Linear Regressor Model
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)
lin_preds = lin_reg.predict(housing_prepared)
# lin_rmse = root_mean_squared_error(housing_labels, lin_preds,)
lin_rmse = cross_val_score(lin_reg, housing_prepared, housing_labels, scoring="neg_mean_squared_error", cv=10)

print(f"The Root Mean Square Error for Linear Regression is {pd.Series(lin_rmse).describe()}")

# Decision Tree Regressor Model
dec_reg = DecisionTreeRegressor()
dec_reg.fit(housing_prepared, housing_labels)
# dec_rmse = root_mean_squared_error(housing_labels, dec_preds,)
dec_preds = dec_reg.predict(housing_prepared)

dec_tree_rmse = cross_val_score(dec_reg, housing_prepared, housing_labels, scoring="neg_mean_squared_error", cv=10)
# print(f"The Root Mean Square Error for Decision Tree Regression is {dec_tree_rmse}")
print(f"The Root Mean Square Error for Decision Tree Regression is{pd.Series(dec_tree_rmse).describe()}")

# random Forest Regressor Model
random_forest_reg = RandomForestRegressor()
random_forest_reg.fit(housing_prepared, housing_labels)
random_forest_preds = random_forest_reg.predict(housing_prepared)
# random_forest_rmse = root_mean_squared_error(housing_labels, random_forest_preds,)
random_forest_rmse = cross_val_score(random_forest_reg, housing_prepared, housing_labels, scoring="neg_mean_squared_error", cv=10)

print(f"The Root Mean Square Error for Random forest Regression is {pd.Series(random_forest_rmse).describe()}")


[[-1.65044886  1.3366276  -0.12821464 ...  0.          0.
   0.        ]
 [ 0.6101325  -0.7499318   1.62466495 ...  0.          0.
   0.        ]
 [-1.4258878   1.01381909  1.86369398 ...  0.          1.
   0.        ]
 ...
 [-1.20631698  0.74715118  1.38563591 ...  0.          0.
   0.        ]
 [ 0.7398789  -0.81542918 -0.20789099 ...  0.          0.
   0.        ]
 [ 0.68997644 -0.85285626  0.11081439 ...  0.          0.
   1.        ]]
The Root Mean Square Error for Linear Regression is count    1.000000e+01
mean    -4.764562e+09
std      1.969248e+08
min     -5.146848e+09
25%     -4.844516e+09
50%     -4.770828e+09
75%     -4.693566e+09
max     -4.409010e+09
dtype: float64
The Root Mean Square Error for Decision Tree Regression iscount    1.000000e+01
mean    -4.619196e+09
std      2.975356e+08
min     -4.984249e+09
25%     -4.795560e+09
50%     -4.658598e+09
75%     -4.403389e+09
max     -4.064519e+09
dtype: float64
The Root Mean Square Error for Random forest Regression is count